In [6]:
from __future__ import annotations

"""
Scaffold + duplicate leakage checker for k folds + one holdout set.

What it does:
- Loads folds from explicit paths (via a dict in this file, no CLI patterns).
- Loads a holdout CSV from an explicit path.
- Standardizes each SMILES (best effort, configurable):
  1) parse smiles
  2) optional: keep largest fragment (desalt / mixtures)
  3) optional: uncharge (decharge) using RDKit MolStandardize Uncharger if available
  4) optional: tautomer canonicalization using RDKit MolStandardize TautomerEnumerator if available
  5) canonical smiles
- Computes Bemis-Murcko scaffold SMILES from the standardized parent.
- Checks duplicates by standardized canonical smiles:
  - within each split
  - across splits
- Checks scaffold overlap:
  - pairwise across folds (compact matrix + compact list)
  - holdout vs each fold (row in matrix + compact list)

Assumptions:
- Each CSV has at least a "smiles" column (case-insensitive fallback supported).
- Other columns (pIC50 etc) are ignored for leakage checks.

Changelog:
- v2.2: Removed argparse and replaced with in-code split_paths dict.
- v2.2: Added toggles for desalting, decharging, tautomer canonicalization.
- v2.2: Added counts for tautomer canonicalization changes (best-effort).
- v2.3: Added compact leakage tables (scaffold overlap matrix and holdout overlap row).
- v2.3: Pairwise overlap printouts made compact (counts only + optional examples for top pairs).
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

# Optional standardization utilities (available in most RDKit builds)
try:
    from rdkit.Chem.MolStandardize import rdMolStandardize
except Exception:
    rdMolStandardize = None


# =============================
# User configuration
# =============================

CONFIG: Dict[str, object] = {
    "split_paths": {
        "fold_0": r"combination_1300_molecules_and_0_%_synthetic\fold_0.csv",
        "fold_1": r"combination_1300_molecules_and_0_%_synthetic\fold_1.csv",
        "fold_2": r"combination_1300_molecules_and_0_%_synthetic\fold_2.csv",
        "fold_3": r"combination_1300_molecules_and_0_%_synthetic\fold_3.csv",
        "fold_4": r"combination_1300_molecules_and_0_%_synthetic\fold_4.csv",
        "holdout": r"heldout_datasets\heldout_testset.csv",
    },
    "standardize": {
        "desalt": True,  # keep largest fragment
        "decharge": True,  # RDKit Uncharger (if available)
        "tautomer_canonicalize": False,  # RDKit TautomerEnumerator (if available)
    },
    "printing": {
        "max_examples": 15,  # examples to print for duplicates and for top overlap pairs
        "top_overlap_pairs_to_show_examples": 5,  # show example scaffolds for top N overlapping fold-pairs
    },
}


# =============================
# Standardization helpers
# =============================

_UNCHARGER = None
_TAUT_ENUM = None

if rdMolStandardize is not None:
    try:
        _UNCHARGER = rdMolStandardize.Uncharger()
    except Exception:
        _UNCHARGER = None

    try:
        _TAUT_ENUM = rdMolStandardize.TautomerEnumerator()
    except Exception:
        _TAUT_ENUM = None


def _largest_fragment(mol: Chem.Mol) -> Tuple[Chem.Mol, bool]:
    """
    Keep the largest fragment by heavy atom count.
    Returns (mol, stripped_flag).
    """
    try:
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    except Exception:
        return mol, False

    if not frags or len(frags) <= 1:
        return mol, False

    best = max(frags, key=lambda m: int(m.GetNumHeavyAtoms()))
    return best, True


def _safe_cansmi(mol: Chem.Mol) -> Optional[str]:
    """
    Canonical SMILES from mol. Returns None on failure.
    """
    if mol is None:
        return None
    try:
        s = Chem.MolToSmiles(mol, canonical=True)
        return s if s else None
    except Exception:
        return None


def standardize_smiles(
    smiles: str,
    *,
    desalt: bool,
    decharge: bool,
    tautomer_canonicalize: bool,
) -> Tuple[Optional[str], Optional[Chem.Mol], Dict[str, int]]:
    """
    Best-effort standardization:
    - Parse
    - Optional desalt: keep largest fragment
    - Optional decharge: RDKit Uncharger (if available)
    - Optional tautomer canonicalization: RDKit TautomerEnumerator (if available)
    - Canonicalize SMILES

    Returns:
      (canonical_smiles, mol, info)

    info keys:
      parsed, salt_stripped, decharged, tautomer_canonicalized
    """
    info = {
        "parsed": 0,
        "salt_stripped": 0,
        "decharged": 0,
        "tautomer_canonicalized": 0,
    }

    if smiles is None:
        return None, None, info

    s0 = str(smiles).strip()
    if not s0:
        return None, None, info

    mol = Chem.MolFromSmiles(s0)
    if mol is None:
        return None, None, info
    info["parsed"] = 1

    if desalt:
        mol2, stripped = _largest_fragment(mol)
        mol = mol2
        if stripped:
            info["salt_stripped"] = 1

    if decharge and _UNCHARGER is not None:
        try:
            before = _safe_cansmi(mol)
            mol2 = _UNCHARGER.uncharge(mol)
            after = _safe_cansmi(mol2)
            if before is not None and after is not None and before != after:
                info["decharged"] = 1
            mol = mol2
        except Exception:
            pass

    if tautomer_canonicalize and _TAUT_ENUM is not None:
        try:
            before = _safe_cansmi(mol)
            mol2 = _TAUT_ENUM.Canonicalize(mol)
            after = _safe_cansmi(mol2)
            if before is not None and after is not None and before != after:
                info["tautomer_canonicalized"] = 1
            mol = mol2
        except Exception:
            pass

    can = _safe_cansmi(mol)
    if can is None:
        return None, None, info

    return can, mol, info


def bemis_murcko_scaffold_smiles(mol: Chem.Mol) -> Optional[str]:
    """
    Returns Bemis-Murcko scaffold SMILES for a molecule.
    Uses RDKit MurckoScaffold on an already-standardized mol.
    """
    if mol is None:
        return None
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        s = Chem.MolToSmiles(scaf, canonical=True)
        return s if s else None
    except Exception:
        return None


# =============================
# Loading + bookkeeping
# =============================

def _find_smiles_column(df: pd.DataFrame) -> str:
    """
    Find a smiles column, case-insensitive.
    Raises if none found.
    """
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}
    if "smiles" in lower:
        return lower["smiles"]

    for cand in ["smile", "canonical_smiles", "smiles_string"]:
        if cand in lower:
            return lower[cand]

    raise ValueError(f"No SMILES column found. Columns: {cols}")


@dataclass
class SplitData:
    name: str
    path: Path
    n_rows: int

    valid_count: int
    invalid_count: int

    salt_stripped_count: int
    decharged_count: int
    tautomer_canonicalized_count: int

    canonical_smiles: List[str]
    scaffolds: List[str]

    smiles_set: Set[str]
    scaffold_set: Set[str]

    dup_smiles: Dict[str, int]
    dup_scaffolds: Dict[str, int]


def load_split(
    path: Path,
    name: str,
    *,
    desalt: bool,
    decharge: bool,
    tautomer_canonicalize: bool,
) -> SplitData:
    df = pd.read_csv(path)
    smi_col = _find_smiles_column(df)
    smiles_raw = df[smi_col].astype(str).tolist()

    canon: List[str] = []
    scaff: List[str] = []

    valid = 0
    invalid = 0
    salt_stripped = 0
    decharged_count = 0
    taut_count = 0

    for s in smiles_raw:
        can, mol, info = standardize_smiles(
            s,
            desalt=desalt,
            decharge=decharge,
            tautomer_canonicalize=tautomer_canonicalize,
        )
        if can is None or mol is None:
            invalid += 1
            continue

        valid += 1
        salt_stripped += int(info.get("salt_stripped", 0))
        decharged_count += int(info.get("decharged", 0))
        taut_count += int(info.get("tautomer_canonicalized", 0))

        canon.append(can)
        sc = bemis_murcko_scaffold_smiles(mol)
        scaff.append(sc if sc is not None else "<NO_SCAFFOLD>")

    dup_smiles: Dict[str, int] = {}
    for x in canon:
        dup_smiles[x] = dup_smiles.get(x, 0) + 1
    dup_smiles = {k: v for k, v in dup_smiles.items() if v > 1}

    dup_scaffolds: Dict[str, int] = {}
    for x in scaff:
        dup_scaffolds[x] = dup_scaffolds.get(x, 0) + 1
    dup_scaffolds = {k: v for k, v in dup_scaffolds.items() if v > 1}

    return SplitData(
        name=name,
        path=path,
        n_rows=len(smiles_raw),
        valid_count=valid,
        invalid_count=invalid,
        salt_stripped_count=salt_stripped,
        decharged_count=decharged_count,
        tautomer_canonicalized_count=taut_count,
        canonical_smiles=canon,
        scaffolds=scaff,
        smiles_set=set(canon),
        scaffold_set=set(scaff),
        dup_smiles=dup_smiles,
        dup_scaffolds=dup_scaffolds,
    )


def _print_top_counts(title: str, d: Dict[str, int], max_examples: int) -> None:
    if not d:
        print(f"{title}: 0")
        return
    print(f"{title}: {len(d)} unique keys duplicated")
    items = sorted(d.items(), key=lambda kv: (-kv[1], kv[0]))
    for k, v in items[:max_examples]:
        print(f"  x{v}: {k}")
    if len(items) > max_examples:
        print(f"  ... ({len(items) - max_examples} more)")


def _intersection_examples(a: Set[str], b: Set[str], max_examples: int) -> List[str]:
    inter = sorted(list(a.intersection(b)))
    return inter[:max_examples]


def _format_int_cell(x: int, width: int) -> str:
    """
    Right-aligned fixed-width integer cell as string.
    """
    return str(int(x)).rjust(width)


def _print_matrix(title: str, labels: List[str], mat: List[List[int]]) -> None:
    """
    Print a compact square matrix with row/col labels.
    """
    if not labels:
        return

    max_label = max(len(x) for x in labels)
    max_val = 0
    for row in mat:
        for v in row:
            max_val = max(max_val, int(v))
    cell_w = max(3, len(str(max_val)))

    print(title)
    header = " " * (max_label + 2) + " ".join(x.rjust(cell_w) for x in labels)
    print(header)
    for i, lab in enumerate(labels):
        row = " ".join(_format_int_cell(mat[i][j], cell_w) for j in range(len(labels)))
        print(lab.ljust(max_label) + "  " + row)
    print()


def _print_vector_row(title: str, labels: List[str], values: List[int]) -> None:
    """
    Print a compact single-row table with column labels.
    """
    if not labels:
        return

    max_val = max(int(v) for v in values) if values else 0
    cell_w = max(3, len(str(max_val)))
    print(title)
    header = " " * 10 + " ".join(x.rjust(cell_w) for x in labels)
    row = "holdout".ljust(10) + " " + " ".join(_format_int_cell(v, cell_w) for v in values)
    print(header)
    print(row)
    print()


# =============================
# Main report
# =============================

def main() -> None:
    split_paths: Dict[str, str] = CONFIG["split_paths"]  # type: ignore[assignment]
    std_cfg: Dict[str, bool] = CONFIG["standardize"]  # type: ignore[assignment]
    print_cfg: Dict[str, int] = CONFIG["printing"]  # type: ignore[assignment]

    max_examples: int = int(print_cfg.get("max_examples", 15))
    top_pairs_examples: int = int(print_cfg.get("top_overlap_pairs_to_show_examples", 5))

    desalt: bool = bool(std_cfg.get("desalt", True))
    decharge: bool = bool(std_cfg.get("decharge", True))
    tautomer_canonicalize: bool = bool(std_cfg.get("tautomer_canonicalize", True))

    if decharge and _UNCHARGER is None:
        print("[WARN] decharge=True but RDKit Uncharger is not available in this RDKit build. Skipping decharge.")
        decharge = False
    if tautomer_canonicalize and _TAUT_ENUM is None:
        print("[WARN] tautomer_canonicalize=True but RDKit TautomerEnumerator is not available. Skipping tautomer step.")
        tautomer_canonicalize = False

    if "holdout" not in split_paths:
        raise ValueError('CONFIG["split_paths"] must contain a "holdout" entry.')

    holdout_path = Path(split_paths["holdout"])
    fold_items = [(k, v) for k, v in split_paths.items() if k != "holdout"]
    fold_items = sorted(fold_items, key=lambda kv: kv[0])

    for name, p in fold_items:
        if not Path(p).exists():
            raise FileNotFoundError(f"Fold file does not exist: {name} -> {p}")
    if not holdout_path.exists():
        raise FileNotFoundError(f"Holdout file does not exist: holdout -> {holdout_path}")

    print("========================================")
    print("Config")
    print("========================================")
    print(f"desalt: {desalt}")
    print(f"decharge: {decharge}")
    print(f"tautomer_canonicalize: {tautomer_canonicalize}")
    print(f"max_examples: {max_examples}")
    print(f"top_overlap_pairs_to_show_examples: {top_pairs_examples}")
    print()

    print("========================================")
    print("Loading splits + standardizing molecules")
    print("========================================")
    folds: List[SplitData] = []
    for name, p in fold_items:
        path = Path(p)
        print(f"- loading {name}: {path}")
        folds.append(
            load_split(
                path,
                name=name,
                desalt=desalt,
                decharge=decharge,
                tautomer_canonicalize=tautomer_canonicalize,
            )
        )

    print(f"- loading holdout: {holdout_path}")
    holdout = load_split(
        holdout_path,
        name="holdout",
        desalt=desalt,
        decharge=decharge,
        tautomer_canonicalize=tautomer_canonicalize,
    )

    print()
    print("========================================")
    print("Per-split summary")
    print("========================================")
    for s in folds + [holdout]:
        print(f"[{s.name}]")
        print(f"  file: {s.path}")
        print(f"  rows: {s.n_rows}")
        print(f"  valid standardized: {s.valid_count}")
        print(f"  invalid (failed parse/standardize): {s.invalid_count}")
        print(f"  salt stripped count (largest fragment kept): {s.salt_stripped_count}")
        print(f"  decharged count (uncharger changed smiles): {s.decharged_count}")
        print(f"  tautomer canonicalized count (canonical tautomer changed smiles): {s.tautomer_canonicalized_count}")
        print(f"  unique standardized smiles: {len(s.smiles_set)}")
        print(f"  unique scaffolds: {len(s.scaffold_set)}")
        _print_top_counts("  duplicate standardized smiles within split", s.dup_smiles, max_examples=max_examples)
        _print_top_counts("  duplicate scaffolds within split", s.dup_scaffolds, max_examples=max_examples)
        print()

    # -----------------------------
    # Duplicates across all splits
    # -----------------------------
    print("========================================")
    print("Duplicates across splits (standardized smiles)")
    print("========================================")
    all_by_smiles: Dict[str, List[str]] = {}
    for s in folds + [holdout]:
        for can in s.smiles_set:
            all_by_smiles.setdefault(can, []).append(s.name)

    cross_dup = {k: v for k, v in all_by_smiles.items() if len(set(v)) > 1}
    if not cross_dup:
        print("No standardized-smiles duplicates found across different splits.")
    else:
        print(f"Found {len(cross_dup)} standardized-smiles that appear in more than one split.")
        shown = 0
        for smi, where in sorted(cross_dup.items(), key=lambda kv: (-len(set(kv[1])), kv[0])):
            where_u = sorted(list(set(where)))
            print(f"  {smi}  -> {where_u}")
            shown += 1
            if shown >= max_examples:
                break
        if len(cross_dup) > max_examples:
            print(f"  ... ({len(cross_dup) - max_examples} more)")
    print()

    # -----------------------------
    # Scaffold overlap tables
    # -----------------------------
    fold_labels = [s.name for s in folds]
    n = len(folds)

    # Matrix: fold i vs fold j overlap counts
    overlap_mat: List[List[int]] = [[0 for _ in range(n)] for _ in range(n)]
    pair_details: List[Tuple[int, str, str, Set[str]]] = []

    for i in range(n):
        for j in range(n):
            if i == j:
                overlap_mat[i][j] = len(folds[i].scaffold_set)
            else:
                inter = folds[i].scaffold_set.intersection(folds[j].scaffold_set)
                overlap_mat[i][j] = len(inter)
                if i < j:
                    pair_details.append((len(inter), folds[i].name, folds[j].name, inter))

    _print_matrix(
        title="========================================\nFold-vs-fold scaffold overlap counts (diagonal = unique scaffolds in fold)\n========================================",
        labels=fold_labels,
        mat=overlap_mat,
    )

    # Holdout vs folds row
    holdout_vs: List[int] = []
    for f in folds:
        holdout_vs.append(len(holdout.scaffold_set.intersection(f.scaffold_set)))

    _print_vector_row(
        title="========================================\nHoldout-vs-fold scaffold overlap counts\n========================================",
        labels=fold_labels,
        values=holdout_vs,
    )

    # Compact list of only the overlapping pairs (counts only)
    print("========================================")
    print("Compact list of scaffold overlaps (non-zero only)")
    print("========================================")
    nonzero_pairs = [(cnt, a, b, inter) for (cnt, a, b, inter) in pair_details if cnt > 0]
    if not nonzero_pairs:
        print("No fold-vs-fold scaffold overlaps.")
    else:
        nonzero_pairs_sorted = sorted(nonzero_pairs, key=lambda x: (-x[0], x[1], x[2]))
        for cnt, a, b, _ in nonzero_pairs_sorted:
            print(f"{a} vs {b}: {cnt}")
    print()

    # Compact holdout overlaps list
    print("========================================")
    print("Holdout overlaps (non-zero only)")
    print("========================================")
    any_holdout = False
    for f, cnt in zip(folds, holdout_vs):
        if cnt > 0:
            any_holdout = True
            print(f"holdout vs {f.name}: {cnt}")
    if not any_holdout:
        print("No holdout-vs-fold scaffold overlaps.")
    print()

    # Optional examples for the top overlapping fold pairs
    if top_pairs_examples > 0:
        print("========================================")
        print(f"Example shared scaffolds for top {top_pairs_examples} fold pairs (by overlap count)")
        print("========================================")
        nonzero_pairs_sorted = sorted(nonzero_pairs, key=lambda x: (-x[0], x[1], x[2]))
        shown_pairs = 0
        for cnt, a, b, inter in nonzero_pairs_sorted:
            if shown_pairs >= top_pairs_examples:
                break
            ex = sorted(list(inter))[:max_examples]
            print(f"{a} vs {b}: {cnt} shared scaffolds (showing up to {max_examples})")
            for s in ex:
                print(f"  {s}")
            if cnt > max_examples:
                print(f"  ... ({cnt - max_examples} more)")
            print()
            shown_pairs += 1

    print("========================================")
    print("Interpretation")
    print("========================================")
    print("Potential leakage indicators:")
    print("- standardized-smiles duplicates across different splits")
    print("- scaffold overlaps between holdout and any fold (should be 0 for strict holdout)")
    print("- scaffold overlaps between folds (should be 0 if folds were constructed disjoint by scaffold)")
    print()
    print("Notes:")
    print("- Tautomer canonicalization can merge molecules that differ only by tautomer form.")
    print("- Turning tautomer canonicalization on usually makes leakage checks stricter.")


if __name__ == "__main__":
    main()

Config
desalt: True
decharge: True
tautomer_canonicalize: False
max_examples: 15
top_overlap_pairs_to_show_examples: 5

Loading splits + standardizing molecules
- loading fold_0: combination_1300_molecules_and_0_%_synthetic\fold_0.csv
- loading fold_1: combination_1300_molecules_and_0_%_synthetic\fold_1.csv
- loading fold_2: combination_1300_molecules_and_0_%_synthetic\fold_2.csv
- loading fold_3: combination_1300_molecules_and_0_%_synthetic\fold_3.csv
- loading fold_4: combination_1300_molecules_and_0_%_synthetic\fold_4.csv
- loading holdout: heldout_datasets\heldout_testset.csv

Per-split summary
[fold_0]
  file: combination_1300_molecules_and_0_%_synthetic\fold_0.csv
  rows: 260
  valid standardized: 260
  invalid (failed parse/standardize): 0
  salt stripped count (largest fragment kept): 0
  decharged count (uncharger changed smiles): 0
  tautomer canonicalized count (canonical tautomer changed smiles): 0
  unique standardized smiles: 260
  unique scaffolds: 182
  duplicate standa